# VCuboidFIT — Colab chạy Tầng 1 thay cho máy của bạn
Máy bạn chạy web + worker (không cần GPU). Notebook này dùng **GPU miễn phí của Colab** để train PointPillars,
rồi tự gửi kết quả Tầng 1 về worker — web tự mở khoá "Lạ với model" và "Model chưa chắc chắn".

**Chỉ 3 bước:**
1. Trên máy bạn: chạy worker có token và mở tunnel (`docs/colab-dev-setup.md`), lấy **URL** + **TOKEN**.
2. Điền URL + TOKEN vào ô 1 bên dưới (Runtime đã đặt sẵn GPU T4; nếu không, *Runtime → Change runtime type → T4 GPU*).
3. Bấm **Runtime → Run all**. Lần đầu cài mất ~10–15 phút, sau đó notebook chờ việc. Trên web bấm
   **"Chạy Tầng 1 trên Colab"**; khi log in `da gui ket qua ...` là xong.

In [ ]:
# @title 1. Điền URL tunnel và TOKEN của worker
SERVER_URL = ""  # @param {type:"string"}
TOKEN = ""  # @param {type:"string"}
# Để trống TOKEN thì notebook sẽ hỏi (ô nhập không hiện ký tự).

In [ ]:
# @title 2. Kiểm GPU + kết nối worker (dừng ngay nếu sai, khỏi chờ cài đặt)
import json, os, shutil, subprocess, urllib.error, urllib.request
from getpass import getpass

SERVER = SERVER_URL.strip().rstrip("/")
if not SERVER.startswith("https://") or "xxxx" in SERVER or "<" in SERVER:
    raise RuntimeError("❌ SERVER_URL chưa đúng: dán URL THẬT do cloudflared in ra trên máy chạy worker, "
                     "dạng https://<tên>.trycloudflare.com")
if not shutil.which("nvidia-smi") or subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode:
    raise RuntimeError("❌ Chưa có GPU: Runtime → Change runtime type → T4 GPU → Save, rồi Run all lại.")
print("GPU:", subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip())

def call(path, token=None):
    req = urllib.request.Request(SERVER + path, headers={"Authorization": f"Bearer {token}"} if token else {})
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            return r.status, r.read().decode()
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode()
    except Exception as e:
        raise RuntimeError(f"❌ Không gọi được {SERVER} ({e}). Kiểm cloudflared còn chạy, URL đúng; đợi 30 s rồi Run all lại.")

status, body = call("/health")
if status != 200:
    raise RuntimeError(f"❌ {SERVER}/health trả {status}: {body[:200]}")
print("Worker:", body)
TOKEN_VALUE = TOKEN.strip() or getpass("TOKEN của worker (dán vào, không hiện ký tự): ").strip()
status, body = call("/remote/t1/t1_000000000000/bundle", TOKEN_VALUE)   # mã việc giả: chỉ kiểm token, không nhận việc
if status == 401:
    raise RuntimeError("❌ Sai TOKEN: phải đúng chuỗi VCF_REMOTE_TOKEN đã đặt khi chạy worker.")
if "remote_disabled" in body:
    raise RuntimeError("❌ Worker chưa bật cầu nối: đặt VCF_REMOTE_TOKEN rồi khởi động lại worker.")
os.environ["VCF_REMOTE_TOKEN"] = TOKEN_VALUE
print("✅ Kết nối + token OK")

In [ ]:
# @title 3. Cài môi trường (lần đầu ~10–15 phút)
import subprocess, sys

def run(cmd, **kw):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **kw)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"❌ Lỗi khi chạy: {' '.join(cmd)} (exit {p.returncode}). Xem log phía trên.")

REPO = "/content/VCuboidFit"
if os.path.isdir(REPO + "/.git"):
    run(["git", "-C", REPO, "pull", "-q"])
else:
    run(["git", "clone", "-q", "--depth", "1", "https://github.com/Nguyendev037/VCuboidFit.git", REPO])
os.environ["VCF_HOME"] = "/content/vcf"
run(["bash", REPO + "/model/scripts/colab_setup.sh"])
os.environ["PCDET_ROOT"] = "/content/vcf/OpenPCDet"
os.environ["PYTHONPATH"] = REPO + "/model/worker:" + os.environ["PCDET_ROOT"]
print("✅ Cài xong")

In [ ]:
# @title 4. Chạy: chờ việc từ web → train + suy luận → gửi kết quả (bấm ■ để dừng)
print("Đang chờ việc. Trên web: Tham số nâng cao → 'Chạy Tầng 1 trên Colab'.")
print("Log: 'khong co viec' = đang chờ · 'nhan viec' = bắt đầu · 'da gui ket qua' = XONG.")
run([sys.executable, "-u", REPO + "/model/scripts/colab_agent.py", "--server", SERVER, "--work", "/content/vcf"],
    env=os.environ)